# 18 — Agent and Multi-Agent Prompt Contracts

## Scenario
We are building a Northstar assistant that can answer questions about the company's database and summarize long company policies.

**The Problem:** Writing one massive "God Prompt" instructing the model to be a Database Expert AND a Summarization Expert is brittle and leads to hallucinations. 

**The Solution:** We use a **Multi-Agent** architecture. We create a central "Supervisor" agent that delegates tasks to narrow "Specialist" agents using strict Pydantic contracts.

In [ ]:
import os
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'


## Step 1: Defining the Contracts (Schemas)

Instead of telling the Supervisor to "talk to the database agent", we define strict Pydantic schemas. The Supervisor *must* generate a valid JSON payload matching this schema to initiate a handoff.

In [ ]:
class DatabaseQueryContract(BaseModel):
    table_name: str = Field(description="The exact name of the table to query.")
    sql_query: str = Field(description="A valid SQL SELECT statement.")

class SummarizationContract(BaseModel):
    text_to_summarize: str = Field(description="The raw text that needs summarizing.")
    max_words: int = Field(description="Maximum length of the summary.")


## Step 2: The Specialist Agents

In a real system, these would be separate LLM calls or actual Python functions. Here, we define them as Python functions that accept our strict Contracts.

In [ ]:
def execute_database_agent(contract: DatabaseQueryContract):
    print(f"[DATABASE AGENT] Executing: {contract.sql_query} on table '{contract.table_name}'")
    # Simulated database result
    return "Result: 500 active users."

def execute_summarization_agent(contract: SummarizationContract):
    print(f"[SUMMARY AGENT] Summarizing text to {contract.max_words} words...")
    # Simulated summary result
    return "Result: The policy mandates 30 days notice for cancellation."


## Step 3: The Supervisor Agent

The Supervisor is given tools. The tools are defined by our Pydantic Contracts. This enforces the boundary.

In [ ]:
def supervisor_route(user_prompt):
    print(f"\nUSER: {user_prompt}")
    
    # The Supervisor has no knowledge of how to do the tasks, 
    # it only knows the schemas required to delegate them.
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction="You are a router. Delegate the user's request to the appropriate tool.",
            temperature=0.0,
            # We pass the Pydantic schemas as tools to the model
            tools=[DatabaseQueryContract, SummarizationContract]
        )
    )

    # Enforce the contract at runtime
    if not response.function_calls:
        print("[SUPERVISOR] Handled request directly (No delegation needed).")
        print(response.text)
        return

    for tool_call in response.function_calls:
        if tool_call.name == "DatabaseQueryContract":
            print("[SUPERVISOR] Delegating to Database Agent...")
            # The SDK automatically handles unpacking the JSON arguments into the schema
            contract = DatabaseQueryContract(**tool_call.args)
            result = execute_database_agent(contract)
            print(result)
            
        elif tool_call.name == "SummarizationContract":
            print("[SUPERVISOR] Delegating to Summarization Agent...")
            contract = SummarizationContract(**tool_call.args)
            result = execute_summarization_agent(contract)
            print(result)

supervisor_route("How many active users do we have in the 'users' table?")
supervisor_route("Can you summarize this 50 page document in 10 words? [Document Text...]")


## Conclusion

By designing **Multi-Agent Systems** connected by strict **Pydantic Contracts**, we create software that is testable, deterministic, and scalable. 

Instead of fighting with a single monolithic prompt, we build a state machine where each agent has a single responsibility, and the boundaries between them are enforced by code.